In [2]:
import re
import spacy
import unicodedata

In [5]:
class TextCleaner:
    def __init__(self, use_lemmatization: bool = True):
        self.use_lemmatization = use_lemmatization
        if self.use_lemmatization:
            # Loads a spaCy English language model (en_core_web_sm)
            # Disables the NER (Named Entity Recognition) and parser (Dependency Parsing) components to make it faster and lighter
            self.nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

    def clean_basic_structures(self, text: str):
        if not isinstance(text, str):
            return ""

        # Remove URLs using regex
        text = re.sub(r'https?://\S+|www\.\S+', '', text)

        # Remove HTML tages
        text = re.sub(r'<[^>]*>', '', text)

        # Normalize Unicode characters (e.g., "café" → "cafe") by decomposing
        # handles accents, curly quotes, odd character formatting
        # accented characters into base + diacritic, then remove all non-ASCII
        # characters (accents, special symbols, etc.) to get plain ASCII text
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

        # 4. Collapse multiple white spaces or tabs into a single uniform space
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    def prepare_for_classical_ml(self, text: str):
        # lowercase basic cleaned
        cleaned = self.clean_basic_structures(text).lower()
        # Strip all punctuation and special symbols except basic text characters and numbers
        cleaned = re.sub(r'[^a-zA-Z0-9\s]', '', cleaned)
        # Run through spaCy pipeline to compute lemmas and analyze syntax tags
        doc = self.nlp(cleaned)
        # Filter out stopwords (using NLTK/spaCy built-in rules) and white spaces
        # Then grab the base word lemma form (e.g., 'running' -> 'run', 'studies' -> 'study')
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]

        return ' '.join(tokens)

    def pipeline(self, text: str):
        if self.use_lemmatization:
            return self.prepare_for_classical_ml(text)
        else:
            return self.clean_basic_structures(text)

In [6]:
# 2. Define a noisy sample sentence that tests every edge case of your cleaner
sample_news = """
🚨 BREAKING NEWS: The corrupt politicians are running away with the money!!
Check out the full report here: https://fakenewsexposed.com/the-scandal
<p>This is a complete disaster for the country.</p>
"""

print("--- RAW SOURCE TEXT ---")
print(sample_news.strip())
print("\n" + "="*50 + "\n")

# 3. Test Phase 2 Configuration: Classical Machine Learning Path
print("--- TESTING: CLASSICAL ML PATH (use_lemmatization=True) ---")
ml_cleaner = TextCleaner(use_lemmatization=True)
ml_output = ml_cleaner.pipeline(sample_news)
print(f"Result:\n\"{ml_output}\"")
print("\n" + "-"*50 + "\n")

# 4. Test Phase 3 Configuration: Transformer / RoBERTa Path
print("--- TESTING: TRANSFORMER PATH (use_lemmatization=False) ---")
transformer_cleaner = TextCleaner(use_lemmatization=False)
transformer_output = transformer_cleaner.pipeline(sample_news)
print(f"Result:\n\"{transformer_output}\"")

--- RAW SOURCE TEXT ---
🚨 BREAKING NEWS: The corrupt politicians are running away with the money!!
Check out the full report here: https://fakenewsexposed.com/the-scandal
<p>This is a complete disaster for the country.</p>


--- TESTING: CLASSICAL ML PATH (use_lemmatization=True) ---
Result:
"break news corrupt politician run away money check report complete disaster country"

--------------------------------------------------

--- TESTING: TRANSFORMER PATH (use_lemmatization=False) ---
Result:
"BREAKING NEWS: The corrupt politicians are running away with the money!! Check out the full report here: This is a complete disaster for the country."
